# Probabilistic Verification

## What This Is
Deterministic verification proves properties for *all* inputs in a set. Sometimes this is too strong — we may want probabilistic guarantees: with high probability over random inputs, the property holds.

**PAC (Probably Approximately Correct) guarantees**: Given ε (error) and δ (confidence), guarantee that P(property violated) ≤ ε with confidence 1-δ.

**Statistical model checking (SMC)**: Sample trajectories from a stochastic system, check property, apply hypothesis testing (Chernoff bounds, sequential probability ratio test) to bound the violation probability.

**Scenario optimization**: For robust optimization problems, the scenario approach gives explicit sample complexity bounds: N ≥ (2/ε)(ln(1/δ) + n) samples suffice for PAC guarantees with n constraints.

In [ ]:
import numpy as np
from typing import Callable, Tuple

np.random.seed(42)

# Statistical Model Checking for NN output properties

def pac_sample_size(epsilon: float, delta: float) -> int:
    """Hoeffding bound: N samples for P(|mean - true| > eps) <= delta."""
    return int(np.ceil(np.log(2/delta) / (2 * epsilon**2)))

def statistical_mc(property_fn: Callable, sampler: Callable,
                   n_samples: int, confidence: float = 0.95) -> Tuple[float, float, float]:
    """
    Statistical model checking via Hoeffding confidence interval.
    Returns: (estimated_violation_rate, ci_lo, ci_hi)
    """
    violations = sum(1 for _ in range(n_samples) if not property_fn(sampler()))
    rate = violations / n_samples
    
    # Hoeffding confidence interval
    z = np.sqrt(np.log(2 / (1 - confidence)) / (2 * n_samples))
    return rate, max(0, rate - z), min(1, rate + z)

# Small ReLU network
np.random.seed(0)
W1 = np.random.randn(8, 4) * 0.5
b1 = np.random.randn(8) * 0.1
W2 = np.random.randn(2, 8) * 0.5
b2 = np.random.randn(2) * 0.1

def net(x):
    h = np.maximum(0, W1 @ x + b1)
    return W2 @ h + b2

x0 = np.array([0.5, -0.3, 0.8, 0.1])

# Property: output[0] > output[1] (class 0 dominates)
def robustness_property(x): return net(x)[0] > net(x)[1]

# Sampler: uniform from L-inf ball
eps = 0.3
def sample_from_ball(): return x0 + np.random.uniform(-eps, eps, 4)

print('Probabilistic Verification via Statistical Model Checking')
print('=' * 55)
print(f'Property: output[0] > output[1] for x in L-inf({eps}) ball')
print()

for n_samples in [100, 1000, 10000]:
    rate, ci_lo, ci_hi = statistical_mc(robustness_property, sample_from_ball, n_samples)
    print(f'N={n_samples:6d}: violation rate={rate:.4f} 95% CI=[{ci_lo:.4f}, {ci_hi:.4f}]')

print()
for eps_req, delta_req in [(0.05, 0.05), (0.01, 0.05), (0.001, 0.01)]:
    n_req = pac_sample_size(eps_req, delta_req)
    print(f'For eps={eps_req}, delta={delta_req}: need N={n_req:,} samples')
